# **Project Title:**
Prediction market (Polymarket)

by Songqiao Qi

## Project objective:
Analyze open Polymarket activity and evaluate whether pre-announcement trading around the US-Iran ceasefire event shows abnormal wallet trading behavior.

## Data overview:
- Primary data APIs: Polymarket Gamma API and Dune API.
- Analytical APIs: Vertex AI, Claude API, and Azure OpenAI API.
- Timeframe: platform context (open events) and a case window (April 7, 2026).
- Focus of analysis: event-level metrics and wallet-level trade behavior.

<img src="polymarket_project_flow_v6.svg" alt="Polymarket Project Flow" width="760"/>


## D1. Library imports

In [34]:
import json
import os
import time
from pathlib import Path

import google.auth
import numpy as np
import pandas as pd
import requests
from anthropic import Anthropic
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row as bokeh_row
from bokeh.models import ColumnDataSource, CustomJS, HoverTool, Select, Slider
from bokeh.plotting import figure
from dotenv import load_dotenv
from google.auth.transport.requests import Request
from openai import AzureOpenAI

output_notebook()
load_dotenv()

Loading BokehJS ...

True

## Part A: General Polymarket Analysis

### D2. Data Retrieval and Pre Processing

In [2]:
# ask Polymarket gamma endpoint for open active events, paged because gamma caps one request at 500
GAMMA_BASE = "https://gamma-api.polymarket.com"
EVENTS_PAGE_LIMIT = 250
EVENTS_TARGET_ROWS = 12000

all_event_records = []

for offset in range(0, EVENTS_TARGET_ROWS, EVENTS_PAGE_LIMIT):
    response = requests.get(
        f"{GAMMA_BASE}/events",
        params={
            "limit": EVENTS_PAGE_LIMIT,
            "offset": offset,
            "closed": "false",
            "active": "true",
        },
        timeout=45,
    )
    response.raise_for_status()
    records = response.json()
    all_event_records.extend(records)

    if len(records) < EVENTS_PAGE_LIMIT:
        break

In [3]:
# flatten nested event json, inspect one row
events_raw = pd.json_normalize(all_event_records, sep=".")
print(f"Open events fetched: {len(events_raw):,}")
display(events_raw.head(1))

Open events fetched: 10,843


,id,ticker,slug,title,description,resolutionSource,startDate,creationDate,endDate,image,...,finishedTimestamp,teams,eventMetadata.leagueTier,eventMetadata.league,eventMetadata.gameId,eventMetadata.tournament,eventMetadata.serie,tweetCount,eventMetadata.finalPrice,eventMetadata.priceToBeat
0,16167,microstrategy-sell-any-bitcoin-in-2025,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",,2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,2025-12-31T12:00:00Z,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# need to pull category from event tags (nested dict), pd.json_normalize cannot handle this
category_values = []

for _, row in events_raw.iterrows():
    category = "uncategorized"

    for tag in row.get("tags", []) or []:
        label = tag.get("label")
        if label:
            category = label
            break

    category_values.append(category)

events_raw["category"] = category_values

for col in ("volume", "liquidity", "openInterest"):
    events_raw[col] = pd.to_numeric(events_raw[col], errors="coerce")
events_raw["createdAt"] = pd.to_datetime(events_raw["createdAt"], errors="coerce", utc=True)

#### Data profiling and field preparation

In [7]:
# dtype check
display(events_raw.dtypes.to_frame("dtype").T)

,id,ticker,slug,title,description,resolutionSource,startDate,creationDate,endDate,image,...,teams,eventMetadata.leagueTier,eventMetadata.league,eventMetadata.gameId,eventMetadata.tournament,eventMetadata.serie,tweetCount,eventMetadata.finalPrice,eventMetadata.priceToBeat,category
dtype,str,str,str,str,str,str,str,str,str,str,...,object,str,str,str,str,str,float64,float64,float64,str


In [ ]:
# make dollar cols readable
raw_2dp_cols = ["liquidity", "openInterest", "competitive", "volume24hr", "liquidityClob"]
for col in raw_2dp_cols:
    events_raw[col] = pd.to_numeric(events_raw[col], errors="coerce").round(2)

# keep these as usd mn
mn_source_cols = ["volume", "volume1wk", "volume1mo", "volume1yr"]
for col in mn_source_cols:
    events_raw[col] = pd.to_numeric(events_raw[col], errors="coerce")
    events_raw[f"{col}_mn"] = (events_raw[col] / 1_000_000).round(2)

display_cols = [
    "id", "title", "slug",
    "volume_mn", "volume1wk_mn", "volume1mo_mn", "volume1yr_mn",
    "liquidity", "openInterest", "competitive", "volume24hr", "liquidityClob",
    "category"
]
# smaller table for part a
events_part_a = events_raw[display_cols].copy()
display(events_part_a.head(3))

,id,title,slug,volume_mn,volume1wk_mn,volume1mo_mn,volume1yr_mn,liquidity,openInterest,competitive,volume24hr,liquidityClob,category
0,16167,MicroStrategy sells any Bitcoin by ___ ?,microstrategy-sell-any-bitcoin-in-2025,22.28,9.08,15.75,19.56,71163.59,483581.85,0.86,5413.23,71163.59,Finance
1,16183,Kraken IPO by ___ ?,kraken-ipo-in-2025,1.56,0.05,0.29,1.01,1247.12,49576.70,0.93,1329.73,1247.12,exchange
2,16263,Macron out by...?,macron-out-in-2025,1.94,0.05,0.45,1.94,25195.68,71413.52,0.81,2.02,25195.68,France


### D3. Data analysis

#### Question 1
Which are the most actively traded Polymarket events currently (top 2)?

##### Answer
Top 2 open events by one month traded volume (USD mn) as of Apr 30:
- 2026 FIFA World Cup Winner (`id=30615`) — **$446.20M**
- Fed decision in April? (`id=75478`) — **$194.16M**

#### Question 2
What kinds of events are currently being traded?

##### Answer
By monthly traded volume, activity is concentrated in Soccer, Sports, World Elections, etc.

Top categories by 1-month volume (USD mn) as of Apr 30:
- **Soccer:** $517.21M
- **Sports:** $343.58M
- **World Elections:** $293.23M
- **Politics:** $259.14M
- **Economic Policy:** $202.52M

#### Question 3
Can LLM agents be used in prediction-market trading?

##### Answer
**Yes, but mainly as decision-support rather than autonomous execution.**

In this notebook test, three models (Gemini 3 Flash, Claude Sonnet 4.6, GPT 5.3) produced the same pick (**France**) with low confidence (**16/100**). 

Some practical use today:
- event screening and hypothesis generation,
- parsing market metadata and translating it into structured signals,

Why trading agents does not work:
- Trading requires trader to understand market participants' expectation, which is primate information. LLM does not have and will not have enough data to train on.

In [9]:
# rank open events by one month volume
rank_cols = [
    "id", "title", "slug",
    "volume_mn", "volume1wk_mn", "volume1mo_mn", "volume1yr_mn",
    "category"
]
# answer to question 1
print("Top 10 open events by 1-month volume")
display(events_part_a.sort_values("volume1mo_mn", ascending=False)[rank_cols].head(10))

Top 10 open events by 1-month volume


,id,title,slug,volume_mn,volume1wk_mn,volume1mo_mn,volume1yr_mn,category
30,30615,2026 FIFA World Cup Winner,2026-fifa-world-cup-winner-595,939.58,203.95,510.03,930.92,Soccer
33,30829,Democratic Presidential Nominee 2028,democratic-presidential-nominee-2028,1115.37,23.15,175.05,1115.37,World Elections
3141,357625,US x Iran ceasefire extended by...?,us-x-iran-ceasefire-extended-by,176.35,164.62,169.84,169.84,Geopolitics
26,27830,2026 NBA Champion,2026-nba-champion,357.66,37.15,111.07,200.94,Sports
36,31875,Republican Presidential Nominee 2028,republican-presidential-nominee-2028,594.20,15.47,99.79,594.20,Politics
35,31552,Presidential Election Winner 2028,presidential-election-winner-2028,560.87,10.40,87.16,560.87,World Elections
609,98369,Eurovision Winner 2026,eurovision-winner-2026,120.68,13.38,69.83,120.68,Music
621,100371,F1 Drivers' Champion,2026-f1-drivers-champion,136.63,13.10,68.43,136.63,Sports
47,33506,UEFA Champions League Winner,uefa-champions-league-winner,250.08,18.67,64.65,220.29,Soccer
2461,334550,What price will Bitcoin hit in April?,what-price-will-bitcoin-hit-in-april-2026,57.46,17.53,54.54,54.54,Bitcoin


#### Event volumn viz

In [17]:
# chart the top events, with a category filter
chart_cols = ["id", "title", "category", "volume1mo_mn"]
hot_events_all = events_part_a.dropna(subset=["volume1mo_mn"])[chart_cols].copy()
hot_events_all["chart_title"] = hot_events_all["title"].str.slice(0, 80)
hot_events_all["bar_color"] = np.where(hot_events_all["title"].str.contains("2026 FIFA World Cup", case=False, na=False), "#00a65a", "#bfe8bf")

# keep only top 10 overall plus top 10 in each category, otherwuse browser will crash
hot_events_all = pd.concat([
    hot_events_all.sort_values("volume1mo_mn", ascending=False).head(10),
    hot_events_all.sort_values("volume1mo_mn", ascending=False).groupby("category", group_keys=False).head(10),
]).drop_duplicates("id").reset_index(drop=True)

hot_events = hot_events_all.sort_values("volume1mo_mn", ascending=False).head(10).sort_values("volume1mo_mn")
source = ColumnDataSource(hot_events)
full_source = ColumnDataSource(hot_events_all)
category_select = Select(title="Category", value="All", options=["All"] + sorted(hot_events_all["category"].dropna().unique().tolist()))

p = figure(
    y_range=hot_events["chart_title"].tolist(),
    height=500,
    width=800,
    title="What’s Hot on Polymarket?",
    toolbar_location=None,
)
p.hbar(y="chart_title", right="volume1mo_mn", height=0.65, source=source, color="bar_color")
p.add_tools(HoverTool(tooltips=[("Event ID", "@id"), ("Title", "@title"), ("Category", "@category"), ("1-month volume", "@volume1mo_mn{0,0.00} mn")]))
p.xaxis.axis_label = "1-Month Volume (USD mn)"
p.yaxis.axis_label = "Event Title"
p.grid.grid_line_alpha = 0.25

# widget required javascript
callback = CustomJS(args=dict(source=source, full_source=full_source, p=p, category_select=category_select), code="""
    const full = full_source.data, selected = category_select.value;
    const rows = [];

    for (let i = 0; i < full.title.length; i++) {
        if (selected === 'All' || full.category[i] === selected) rows.push(i);
    }

    rows.sort((a, b) => full.volume1mo_mn[b] - full.volume1mo_mn[a]);
    const top = rows.slice(0, 10).reverse();
    const data = {};
    for (const k in full) data[k] = top.map(i => full[k][i]);

    source.data = data;
    p.y_range.factors = data.chart_title;
""")
category_select.js_on_change("value", callback)
show(column(category_select, p))

#### Category volumn viz

In [ ]:
# aggr one month volume by category
category_volume = (
    events_part_a
    .groupby("category", as_index=False)["volume1mo_mn"]
    .sum()
    .sort_values("volume1mo_mn", ascending=False)
    .head(10)
)
category_volume = category_volume.sort_values("volume1mo_mn")

source = ColumnDataSource(category_volume)
p = figure(
    y_range=category_volume["category"].tolist(),
    height=450,
    width=800,
    title="Where Is the 1-Month Volume Concentrated?",
    toolbar_location=None,
)
p.hbar(y="category", right="volume1mo_mn", height=0.65, source=source, color="#bfe8bf")
p.add_tools(HoverTool(tooltips=[("Category", "@category"), ("1-month volume", "@volume1mo_mn{0,0.00} mn")]))
p.xaxis.axis_label = "1-Month Volume (USD mn)"
p.yaxis.axis_label = "Category"
p.grid.grid_line_alpha = 0.25
show(p)

#### save outputs

In [19]:
# save all open events to csv
output_dir = "./polymarket_outputs"
os.makedirs(output_dir, exist_ok=True)

events_path = os.path.join(output_dir, "events_open_raw.csv")
events_raw.to_csv(events_path, index=False, mode="w")

In [20]:
# test three ai api connections
# vertex need google auth first
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
creds.refresh(Request())

VERTEX_PROJECT_ID = os.getenv("VERTEX_PROJECT_ID")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "global")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-3-flash-preview")
VERTEX_API_HOST = "aiplatform.googleapis.com" if VERTEX_LOCATION == "global" else f"{VERTEX_LOCATION}-aiplatform.googleapis.com"
vertex_endpoint = f"https://{VERTEX_API_HOST}/v1/projects/{VERTEX_PROJECT_ID}/locations/{VERTEX_LOCATION}/publishers/google/models/{VERTEX_MODEL}:generateContent"
vertex_headers = {"Authorization": f"Bearer {creds.token}", "Content-Type": "application/json"}

# claude and azure pull their config from .env
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL")
claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
azure_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT").rstrip("/"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

# same test prompt goes to all three
test_prompt = "Reply OK only."

# call vertex
vertex_payload = {"contents": [{"role": "user", "parts": [{"text": test_prompt}]}], "generationConfig": {"temperature": 0, "maxOutputTokens": 512}}
vertex_resp = requests.post(vertex_endpoint, headers=vertex_headers, json=vertex_payload, timeout=60)
vertex_resp.raise_for_status()
print("Vertex Gemini:", vertex_resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip())

# call claude
claude_msg = claude_client.messages.create(model=CLAUDE_MODEL, max_tokens=512, temperature=0, messages=[{"role": "user", "content": test_prompt}])
print("Claude:", claude_msg.content[0].text.strip())

# call azure gpt
gpt_msg = azure_client.responses.create(model=AZURE_OPENAI_DEPLOYMENT, input=test_prompt, max_output_tokens=512)
print("Azure GPT:", gpt_msg.output_text.strip())

Vertex Gemini: OK
Claude: OK
Azure GPT: OK


In [21]:
# throw one event (2026 FIFA) to ai models, this part compare answers
SELECTED_EVENT_ID = "30615"
row = events_raw[events_raw["id"].astype(str) == SELECTED_EVENT_ID].iloc[0]

display(pd.DataFrame([row])[["id", "title", "slug", "volume_mn", "category"]])

# pack event metadata and markets into one prompt
prompt_text = f"""
You are a sports prediction-market analyst.
Given this Polymarket event and its nested market JSON, predict the most likely champion.
Return only: predicted champion, confidence (0-100).

event_id: {row["id"]}
event_title: {row["title"]}
event_slug: {row["slug"]}
volume_usd_mn: {row["volume_mn"]:,.2f}
description: {str(row.get("description", ""))[:1200]}

markets_nested_json: {json.dumps(row["markets"], ensure_ascii=False, default=str)}
"""

# run same prompt through each model
vertex_payload = {"contents": [{"role": "user", "parts": [{"text": prompt_text}]}], "generationConfig": {"temperature": 0.3, "maxOutputTokens": 2048}}
vertex_resp = requests.post(vertex_endpoint, headers=vertex_headers, json=vertex_payload, timeout=60)
vertex_resp.raise_for_status()
vertex_text = vertex_resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()

claude_msg = claude_client.messages.create(model=CLAUDE_MODEL, max_tokens=2048, temperature=0.3, messages=[{"role": "user", "content": prompt_text}])
claude_text = claude_msg.content[0].text.strip()

gpt_msg = azure_client.responses.create(model=AZURE_OPENAI_DEPLOYMENT, input=prompt_text, max_output_tokens=2048)
gpt_text = gpt_msg.output_text.strip()

# print model outputs
print("\n===== Gemini 3 Flash =====\n")
print(vertex_text)
print("\n===== Claude Sonnet 4.6 =====\n")
print(claude_text)
print("\n===== GPT 5.3 =====\n")
print(gpt_text)

,id,title,slug,volume_mn,category
30,30615,2026 FIFA World Cup Winner,2026-fifa-world-cup-winner-595,939.58,Soccer



===== Gemini 3 Flash =====

France, 16.05

===== Claude Sonnet 4.6 =====

**Predicted champion: France**
**Confidence: 16**

===== GPT 5.3 =====

France, 16


## Part B: Abnormal (potential insider) trading


### D2. Data Retrieval and Pre Processing

In [22]:
# basic setup and api config for the case study
EVENT_SLUG = "us-x-iran-ceasefire-by"
ANCHOR_TS_UTC = pd.Timestamp("2026-04-07T22:32:00Z")
PRE_WINDOW_HOURS = 5
PRE_WINDOW_START_UTC = ANCHOR_TS_UTC - pd.Timedelta(hours=PRE_WINDOW_HOURS)

DUNE_API_BASE = "https://api.dune.com/api/v1"

DUNE_API_KEY = os.getenv("DUNE_API_KEY", "").strip()

OUTPUT_DIR = Path("./polymarket_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Anchor UTC:", ANCHOR_TS_UTC)
print("Pre-window:", PRE_WINDOW_START_UTC, "to", ANCHOR_TS_UTC)

Anchor UTC: 2026-04-07 22:32:00+00:00
Pre-window: 2026-04-07 17:32:00+00:00 to 2026-04-07 22:32:00+00:00


#### Event retrieval and pre-window setup


In [25]:
# pull target event (closed) from gamma, print submarkets total for confirmation
response = requests.get(
    f"{GAMMA_BASE}/events",
    params={"slug": EVENT_SLUG},
    timeout=30,
)
response.raise_for_status()
event = response.json()[0]

print("event_id:", event["id"])
print("event_title:", event["title"])

submarkets = pd.json_normalize(event["markets"], sep=".")

# adjust cols timezone
for c in ["createdAt", "startDate", "endDate", "updatedAt"]:
    submarkets[c] = pd.to_datetime(submarkets[c], errors="coerce", utc=True)

submarkets["volume"] = pd.to_numeric(submarkets["volume"], errors="coerce")
submarkets["volume_mn"] = (submarkets["volume"] / 1_000_000).round(2)
submarkets["launch_time"] = submarkets["startDate"].combine_first(submarkets["createdAt"])
# only keep markets that existed in the window
submarkets_pre_anchor = submarkets[submarkets["launch_time"] <= ANCHOR_TS_UTC].copy()

keep_cols = [
    "id", "slug", "question", "active", "closed",
    "launch_time", "createdAt", "startDate", "endDate",
    "volume_mn", "conditionId", "clobTokenIds",
]

print(f"Submarkets total: {len(submarkets):,}")
print(f"Submarkets pre-anchor: {len(submarkets_pre_anchor):,}")
submarkets_pre_anchor.to_csv(OUTPUT_DIR / "case_ceasefire_selected_submarkets.csv", index=False)

event_id: 236840
event_title: US x Iran ceasefire by...?
Submarkets total: 12
Submarkets pre-anchor: 12


#### Dune API query helpers


In [26]:
# dune setup for sql call below
DUNE_HEADERS = {"X-Dune-Api-Key": DUNE_API_KEY, "Content-Type": "application/json"}
DUNE_PERFORMANCE = "medium"
DUNE_POLL_INTERVAL_SECONDS = 2.0
DUNE_TIMEOUT_SECONDS = 300

#### Event/market resolution


In [27]:
# map polymarket condition ids into dune question/outcome rows
# dune is not like gamma, it runs sql async, so we submit first and later query state
selected_condition_ids = [
    c if c.startswith("0x") else f"0x{c}"
    for c in sorted(submarkets_pre_anchor["conditionId"].dropna().str.lower().unique())
]

condition_sql = ", ".join(f"'{c}'" for c in selected_condition_ids)

# sql ask market_details which outcome row each condition id points to
mapping_sql = f"""
-- this query maps each condition id to its question/outcome row in dune
SELECT DISTINCT
    condition_id,
    event_market_name,
    question,
    token_outcome,
    token_outcome_name,
    unique_key
FROM polymarket_polygon.market_details
WHERE lower(condition_id) IN ({condition_sql})
ORDER BY question, token_outcome
"""

# send sql to dune and get an execution id back
response = requests.post(
    f"{DUNE_API_BASE}/sql/execute",
    headers=DUNE_HEADERS,
    json={"sql": mapping_sql, "performance": DUNE_PERFORMANCE},
    timeout=30,
)
response.raise_for_status()
execution_id = response.json()["execution_id"]

# poll dune until query says completed, then fetch result rows
state = ""
while state not in {"QUERY_STATE_COMPLETED", "QUERY_STATE_COMPLETED_PARTIAL"}:
    time.sleep(DUNE_POLL_INTERVAL_SECONDS)
    status_response = requests.get(f"{DUNE_API_BASE}/execution/{execution_id}/status", headers=DUNE_HEADERS, timeout=30)
    status_response.raise_for_status()
    state = status_response.json()["state"]

results_response = requests.get(
    f"{DUNE_API_BASE}/execution/{execution_id}/results",
    headers=DUNE_HEADERS,
    params={"allow_partial_results": "true", "limit": 20000},
    timeout=60,
)
results_response.raise_for_status()

dune_market_map = pd.DataFrame(results_response.json()["result"]["rows"])

print("Mapped rows:", len(dune_market_map))
display(dune_market_map.head(5))
dune_market_map.to_csv(OUTPUT_DIR / "case_ceasefire_dune_market_map.csv", index=False)

Mapped rows: 24


,condition_id,event_market_name,question,token_outcome,token_outcome_name,unique_key
0,0x0de3f76fe9b60857833856bec4f30d815cdb0c283361...,single market,US x Iran ceasefire by April 10?,No,No-US x Iran ceasefire by April 10?,0x0de3f76fe9b60857833856bec4f30d815cdb0c283361...
1,0x0de3f76fe9b60857833856bec4f30d815cdb0c283361...,single market,US x Iran ceasefire by April 10?,Yes,Yes-US x Iran ceasefire by April 10?,0x0de3f76fe9b60857833856bec4f30d815cdb0c283361...
2,0x773abaa5fe55e5cde51a261f444b7921652a4e059ead...,single market,US x Iran ceasefire by April 15?,No,No-US x Iran ceasefire by April 15?,0x773abaa5fe55e5cde51a261f444b7921652a4e059ead...
3,0x773abaa5fe55e5cde51a261f444b7921652a4e059ead...,single market,US x Iran ceasefire by April 15?,Yes,Yes-US x Iran ceasefire by April 15?,0x773abaa5fe55e5cde51a261f444b7921652a4e059ead...
4,0x80059ff4e694f878c0498f6f3a067ee7ca62dc2fc462...,single market,US x Iran ceasefire by April 30?,No,No-US x Iran ceasefire by April 30?,0x80059ff4e694f878c0498f6f3a067ee7ca62dc2fc462...


### D3. Data analysis

#### Backgourd

Prediction markets have been under fire around allegations of insider trading, there has been extensive news coverage on this issue. See, for example, this FT article: https://www.ft.com/barrier/corporate/7ab81d98-cf9c-401d-b105-208c0bc7a23e

#### Question 4
How can trading data be used to detect abnormal (potential insider-like) activity?

##### Answer
1. Find the event and pre-event window around an anchor timestamp.
2. Filter to relevant markets/questions and a meaningful thresholds (omit small trades, usually reail).
3. Build wallet-level features, most important:
   - total notional,
   - YES-vs-NO directional bias,
4. Enrich with wallet metadata (creation/funding time).
5. Flag combinations such as: newly funded wallet + positioning bia + high notional + clustered timing before event information release.

Case metrics used here:
- pre-window rows: **2,251 trades**
- wallets: **632**
- total notional: **$8.44M**
- concentration: top-1 wallet **25.24%**, top-10 wallets **44.34%** of pre-window notional
- direction counts: **510 NO-biased** wallets vs **122 YES-biased** wallets

In [ ]:
# pull april 7 pre-news trades (above 500) across 12 chosen markets
# convert time to iso strings for dune
start_iso = PRE_WINDOW_START_UTC.strftime("%Y-%m-%dT%H:%M:%SZ")
end_iso = ANCHOR_TS_UTC.strftime("%Y-%m-%dT%H:%M:%SZ")
APRIL7_QUESTION_KEYWORD = "april 7"
condition_sql = ", ".join("'" + str(c).replace("'", "''") + "'" for c in selected_condition_ids)

# sql ask market_trades for pre-news orders in april 7 market
trade_sql = f"""
-- this query pulls sizeable trades before the ceasefire news anchor
SELECT
    date_trunc('month', block_time) AS block_month,
    block_number,
    block_time,
    lower(concat('0x', to_hex(tx_hash))) AS tx_hash,
    evt_index,
    action,
    lower(concat('0x', to_hex(contract_address))) AS contract_address,
    lower(concat('0x', to_hex(condition_id))) AS condition_id,
    event_market_name,
    question,
    token_outcome,
    CAST(asset_id AS VARCHAR) AS asset_id,
    price,
    amount,
    shares,
    lower(concat('0x', to_hex(maker))) AS maker,
    lower(concat('0x', to_hex(taker))) AS taker,
    unique_key,
    token_outcome_name
FROM polymarket_polygon.market_trades
WHERE block_time >= from_iso8601_timestamp('{start_iso}')
  AND block_time < from_iso8601_timestamp('{end_iso}')
  AND lower(concat('0x', to_hex(condition_id))) IN ({condition_sql})
  AND lower(question) LIKE '%{APRIL7_QUESTION_KEYWORD}%'
  AND amount > 500
ORDER BY block_time
"""

# submit the trade query
response = requests.post(
    f"{DUNE_API_BASE}/sql/execute",
    headers=DUNE_HEADERS,
    json={"sql": trade_sql, "performance": DUNE_PERFORMANCE},
    timeout=30,
)
response.raise_for_status()
execution_id = response.json()["execution_id"]

state = ""
while state not in {"QUERY_STATE_COMPLETED", "QUERY_STATE_COMPLETED_PARTIAL"}:
    time.sleep(DUNE_POLL_INTERVAL_SECONDS)
    status_response = requests.get(f"{DUNE_API_BASE}/execution/{execution_id}/status", headers=DUNE_HEADERS, timeout=30)
    status_response.raise_for_status()
    state = status_response.json()["state"]

results_response = requests.get(
    f"{DUNE_API_BASE}/execution/{execution_id}/results",
    headers=DUNE_HEADERS,
    params={"allow_partial_results": "true", "limit": 500000},
    timeout=60,
)
results_response.raise_for_status()
pre_trades = pd.DataFrame(results_response.json()["result"]["rows"])

Scope filter: question contains 'April 7'
Pre-window trade rows: 2251
Window min/max: 2026-04-07 17:32:25+00:00 2026-04-07 22:31:59+00:00


,action,amount,asset_id,block_month,block_number,block_time,condition_id,contract_address,event_market_name,evt_index,maker,price,question,shares,taker,token_outcome,token_outcome_name,tx_hash,unique_key
0,CLOB trade,504.29600,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231554,2026-04-07 17:32:25+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,739,0xbff39b9750aa814a67ecf19a461b0e406bd3a023,0.950104,US x Iran ceasefire by April 7?,530.7800,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,No,No-US x Iran ceasefire by April 7?,0x70c648d926e4ed3313ca3dc57223019ed3543ff47435...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
2,CLOB trade,728.77983,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231587,2026-04-07 17:33:31+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,1069,0xbb39ba863319281427a482c8714ffdf69cb7ee8a,0.951000,US x Iran ceasefire by April 7?,766.3300,0x817997d188ab6a5bc85180d561b727401b044403,No,No-US x Iran ceasefire by April 7?,0x8cf47b24344486a3b776c88a4f0b45be785f02e812ed...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
1,CLOB trade,548.37477,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231587,2026-04-07 17:33:31+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,1085,0x8c76c4787bf236398d2d1a42dd3e8af1393206ac,0.950000,US x Iran ceasefire by April 7?,577.2366,0x817997d188ab6a5bc85180d561b727401b044403,No,No-US x Iran ceasefire by April 7?,0x8cf47b24344486a3b776c88a4f0b45be785f02e812ed...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
3,CLOB trade,2018.21000,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231598,2026-04-07 17:33:53+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,2220,0xa75cb55885d667de343b241d26268ef45981895e,0.951986,US x Iran ceasefire by April 7?,2120.0000,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,No,No-US x Iran ceasefire by April 7?,0xbd91e63654f054702f7f60a889164218cf258ae41f79...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
4,CLOB trade,677.08800,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231626,2026-04-07 17:34:49+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,341,0xa75cb55885d667de343b241d26268ef45981895e,0.960000,US x Iran ceasefire by April 7?,705.3000,0x87852665e53317a02163d4f5970413ac20fadf3f,No,No-US x Iran ceasefire by April 7?,0x3b748b68e60b4e7d14572fd189e407bc2a122d6cfa98...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
5,CLOB trade,706.91400,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231636,2026-04-07 17:35:09+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,2544,0xbb39ba863319281427a482c8714ffdf69cb7ee8a,0.954000,US x Iran ceasefire by April 7?,741.0000,0x57cc0ba8ed5ffc0a42cfeddb8cd4ddc528c39f51,No,No-US x Iran ceasefire by April 7?,0x08ee130b9f9adfe3b567f9e5a16e8761fa044f80be27...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
6,CLOB trade,730.78900,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231636,2026-04-07 17:35:09+00:00,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,single market,2546,0x57cc0ba8ed5ffc0a42cfeddb8cd4ddc528c39f51,0.954033,US x Iran ceasefire by April 7?,766.0000,0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e,No,No-US x Iran ceasefire by April 7?,0x08ee130b9f9adfe3b567f9e5a16e8761fa044f80be27...,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...
8,CLOB trade,527.45000,5519474545307429756090043890835774997878002144...,2026-04-01 00:00:00+00:00,85231672,2026-04-07 17:36:21+00:00,0x4c5701bcde0b8fb7d7f4

Saved: polymarket_outputs/case_ceasefire_prewindow_trades.csv


In [30]:
# clean time and numeric cols before any wallet aggregation
pre_trades["block_month"] = pd.to_datetime(pre_trades["block_month"], errors="coerce", utc=True)
pre_trades["block_time"] = pd.to_datetime(pre_trades["block_time"], errors="coerce", utc=True)
for col in ("block_number", "evt_index", "price", "amount", "shares"):
    pre_trades[col] = pd.to_numeric(pre_trades[col], errors="coerce")

# final small cleanup: valid time/amount and chronological order
pre_trades = pre_trades.dropna(subset=["block_time", "amount"]).sort_values(["block_time", "evt_index"])

print("Pre-window trade rows:", len(pre_trades))
print("Window min/max:", pre_trades["block_time"].min(), pre_trades["block_time"].max())

pre_trades.to_csv(OUTPUT_DIR / "case_ceasefire_prewindow_trades.csv", index=False)

Pre-window trade rows: 2251
Window min/max: 2026-04-07 17:32:25+00:00 2026-04-07 22:31:59+00:00


#### Question 5
Is there insider-like behavior in the US-Iran ceasefire case?

##### Answer
**There are insider-like signals for specific wallets, but not definitive proof.**

Some evidence:
- A wallet (`0x68558d...f655a`) was created and first funded on **2026-04-07** (same day as the announcement). Note that the US-Iran ceasefire was annouced on 2026-04-07 at 22:32:00 UTC.
- In the 5-hour pre-window, this wallet traded one-sided toward **YES** (ceasefire, ex ante).
- This wallet has only traded on this particular event: **303 trades**, **$254.5K** notional, actual profict could be even larger.

Other facts:
- Market activity was broad (**632 wallets**) and total notional was high (**$8.44M**) in the window.
- Most trading activities in the window are NO-biased.
- On-chain analysis alone cannot attribute identity.

In [31]:
# roll maker and taker into one wallet table, then total by wallet
wallet_base_cols = ["block_time", "tx_hash", "condition_id", "question", "token_outcome", "amount"]

# maker and taker are both wallets in the trade, so stack both sides
maker_legs = pre_trades[wallet_base_cols + ["maker"]].rename(columns={"maker": "wallet"})
taker_legs = pre_trades[wallet_base_cols + ["taker"]].rename(columns={"taker": "wallet"})

# drop empty wallets and keep real positive notional only
wallet_legs = pd.concat([maker_legs, taker_legs], ignore_index=True).dropna(subset=["wallet", "amount"])
wallet_legs = wallet_legs[wallet_legs["amount"] > 0].copy()
wallet_legs["token_outcome_norm"] = wallet_legs["token_outcome"].astype(str).str.upper().str.strip().replace({"Y": "YES", "N": "NO"})

# first aggregate total size and timing by wallet
wallet_agg = (
    wallet_legs
    .groupby("wallet", as_index=False)
    .agg(
        total_notional_usd=("amount", "sum"),
        wallet_trade_count=("tx_hash", "count"),
        unique_tx_count=("tx_hash", "nunique"),
        market_count=("condition_id", "nunique"),
        first_trade_time=("block_time", "min"),
        last_trade_time=("block_time", "max"),
    )
    .set_index("wallet")
)

# then split the same notional by yes/no outcome
outcome_totals = (
    wallet_legs
    .pivot_table(index="wallet", columns="token_outcome_norm", values="amount", aggfunc="sum", fill_value=0)
    .rename(columns={"YES": "yes_notional_usd", "NO": "no_notional_usd"})
)

# join yes/no totals back and calculate net direction
wallet_agg = wallet_agg.join(outcome_totals[["yes_notional_usd", "no_notional_usd"]]).fillna(0).reset_index()
wallet_agg["net_yes_minus_no_notional"] = wallet_agg["yes_notional_usd"] - wallet_agg["no_notional_usd"]
wallet_agg["direction_bias"] = np.where(
    wallet_agg["net_yes_minus_no_notional"] > 0, "YES (ceasefire)",
    np.where(wallet_agg["net_yes_minus_no_notional"] < 0, "NO (no ceasefire)", "Neutral / mixed"),
)
wallet_agg = wallet_agg.sort_values("total_notional_usd", ascending=False).reset_index(drop=True)

display_cols = [
    "wallet", "total_notional_usd", "yes_notional_usd", "no_notional_usd",
    "net_yes_minus_no_notional", "direction_bias", "wallet_trade_count", 
]
print("Wallets in pre-window:", len(wallet_agg))
wallet_agg.to_csv(OUTPUT_DIR / "case_ceasefire_wallet_aggregates.csv", index=False)

Wallets in pre-window: 632


In [36]:
# build interactive chart for pre-news wallet bias
# prep chart fields, widget will filter this
wallet_chart_all = wallet_agg.sort_values("total_notional_usd", ascending=False).copy()
wallet_chart_all["rank"] = np.arange(1, len(wallet_chart_all) + 1)
wallet_chart_all["wallet_label"] = wallet_chart_all["rank"].astype(str) + ". " + wallet_chart_all["wallet"].str.slice(0, 8) + "..." + wallet_chart_all["wallet"].str.slice(-4)
wallet_chart_all = wallet_chart_all.assign(
    no_left=-wallet_chart_all["no_notional_usd"],
    no_right=0,
    yes_left=0,
    yes_right=wallet_chart_all["yes_notional_usd"],
    net_left=wallet_chart_all["net_yes_minus_no_notional"].clip(upper=0),
    net_right=wallet_chart_all["net_yes_minus_no_notional"].clip(lower=0),
    net_color=np.where(wallet_chart_all["net_yes_minus_no_notional"] >= 0, "#00a65a", "#d62728"),
)

# initial view is top 30
wallet_chart = wallet_chart_all.head(30)
factors = wallet_chart["wallet_label"].iloc[::-1].tolist()
max_notional = wallet_chart[["yes_notional_usd", "no_notional_usd"]].to_numpy().max()

# bokeh reads from ColumnDataSource
source = ColumnDataSource(wallet_chart)
full_source = ColumnDataSource(wallet_chart_all)
top_n_slider = Slider(start=10, end=100, value=30, step=5, title="Top N wallets")
direction_select = Select(title="Net direction", value="All", options=["All", "YES net", "NO net"])

# three bars share same zero line: no left, yes right, net overlay
p = figure(y_range=factors, x_range=(-max_notional * 1.1, max_notional * 1.1), height=760, width=800, title="Who Bet Before the Ceasefire?", toolbar_location=None)
p.hbar(y="wallet_label", left="no_left", right="no_right", height=0.75, source=source, color="#f4b6b6", legend_label="NO notional")
p.hbar(y="wallet_label", left="yes_left", right="yes_right", height=0.75, source=source, color="#bfe8bf", legend_label="YES notional")
p.hbar(y="wallet_label", left="net_left", right="net_right", height=0.35, source=source, color="net_color", legend_label="Net YES - NO")
p.line(x=[0, 0], y=[factors[0], factors[-1]], line_color="black", line_width=2)
p.add_tools(HoverTool(tooltips=[
    ("Wallet", "@wallet"),
    ("Total notional", "@total_notional_usd{$0,0}"),
    ("YES notional", "@yes_notional_usd{$0,0}"),
    ("NO notional", "@no_notional_usd{$0,0}"),
    ("Net YES - NO", "@net_yes_minus_no_notional{$0,0}"),
    ("Direction", "@direction_bias"),
]))
p.xaxis.axis_label = "Notional USD: NO left, YES right"
p.yaxis.axis_label = "Wallets ranked by total notional"
p.legend.location = "top_right"
p.grid.grid_line_alpha = 0.25

# javascript filter
callback = CustomJS(args=dict(source=source, full_source=full_source, p=p, top_n=top_n_slider, direction=direction_select), code="""
    const full = full_source.data, data = {};
    for (const k in full) data[k] = [];

    for (let i = 0; i < Math.min(top_n.value, full.wallet.length); i++) {
        const net = full.net_yes_minus_no_notional[i];
        const keep_direction = direction.value === 'All' || (direction.value === 'YES net' && net > 0) || (direction.value === 'NO net' && net < 0);
        if (keep_direction) for (const k in full) data[k].push(full[k][i]);
    }

    source.data = data;
    p.y_range.factors = data.wallet_label.slice().reverse();

    const max_val = Math.max(1, ...data.yes_notional_usd, ...data.no_notional_usd);
    p.x_range.start = -max_val * 1.1;
    p.x_range.end = max_val * 1.1;
""")

for widget in [top_n_slider, direction_select]:
    widget.js_on_change("value", callback)

show(column(bokeh_row(top_n_slider, direction_select), p))

In [37]:
# check two suspicious wallets
TARGET_WALLETS = [
    "0x72e4daa9b93fd3786f231c3b73c1bdbc4c48740a",
    "0x68558d37cafd9e6612ab32863f55ccdd798f655a",
]
target_wallet_sql = ", ".join(f"'{w}'" for w in TARGET_WALLETS)

# sql ask user lookup for wallet identity, plus max trade time from market_trades
wallet_info_sql = f"""
WITH wallet_info AS (
    -- this part gets wallet creation/funding info from user lookup
    SELECT DISTINCT
        CASE
            WHEN lower(concat('0x', to_hex(polymarket_wallet))) IN ({target_wallet_sql})
                THEN lower(concat('0x', to_hex(polymarket_wallet)))
            ELSE lower(concat('0x', to_hex(owner)))
        END AS wallet,
        lower(concat('0x', to_hex(polymarket_wallet))) AS polymarket_wallet,
        lower(concat('0x', to_hex(owner))) AS owner,
        wallet_type,
        created_time,
        first_funded_time
    FROM polymarket_polygon.users_address_lookup
    WHERE lower(concat('0x', to_hex(polymarket_wallet))) IN ({target_wallet_sql})
       OR lower(concat('0x', to_hex(owner))) IN ({target_wallet_sql})
),
last_trade AS (
    -- this part only asks for max block_time as final trade time
    SELECT wallet, max(block_time) AS last_trade_time
    FROM (
        SELECT lower(concat('0x', to_hex(maker))) AS wallet, block_time
        FROM polymarket_polygon.market_trades
        WHERE lower(concat('0x', to_hex(maker))) IN ({target_wallet_sql})
        UNION ALL
        SELECT lower(concat('0x', to_hex(taker))), block_time
        FROM polymarket_polygon.market_trades
        WHERE lower(concat('0x', to_hex(taker))) IN ({target_wallet_sql})
    )
    GROUP BY 1
)
SELECT wi.*, lt.last_trade_time
FROM wallet_info wi
LEFT JOIN last_trade lt ON wi.wallet = lt.wallet
"""

response = requests.post(
    f"{DUNE_API_BASE}/sql/execute",
    headers=DUNE_HEADERS,
    json={"sql": wallet_info_sql, "performance": DUNE_PERFORMANCE},
    timeout=30,
)
response.raise_for_status()
execution_id = response.json()["execution_id"]

# polling
TERMINAL_OK = {"QUERY_STATE_COMPLETED", "QUERY_STATE_COMPLETED_PARTIAL"}
TERMINAL_ERR = {"QUERY_STATE_FAILED", "QUERY_STATE_CANCELED", "QUERY_STATE_EXPIRED"}
for _ in range(60):
    time.sleep(DUNE_POLL_INTERVAL_SECONDS)
    state = requests.get(f"{DUNE_API_BASE}/execution/{execution_id}/status", headers=DUNE_HEADERS, timeout=30).json()["state"]
    if state in TERMINAL_OK | TERMINAL_ERR:
        break
if state not in TERMINAL_OK:
    raise RuntimeError(f"Dune query ended with state={state}")

# pull result rows into final two-wallet table
wallet_info = pd.DataFrame(
    requests.get(
        f"{DUNE_API_BASE}/execution/{execution_id}/results",
        headers=DUNE_HEADERS,
        params={"allow_partial_results": "true", "limit": 1000},
        timeout=60,
    ).json()["result"]["rows"]
)

for col in ["created_time", "first_funded_time", "last_trade_time"]:
    wallet_info[col] = pd.to_datetime(wallet_info[col], errors="coerce", utc=True)

display(wallet_info.drop(columns=["polymarket_wallet", "wallet", "wallet_type"]))
wallet_info.to_csv(OUTPUT_DIR / "case_ceasefire_target_wallet_info.csv", index=False)

,created_time,first_funded_time,last_trade_time,owner
0,2025-12-10 15:15:56+00:00,2025-12-10 15:16:46+00:00,2026-04-24 20:01:04+00:00,0xa3c274d558658538f76661e403ad71aaf342f493
1,2026-04-07 13:54:47+00:00,2026-04-07 13:59:17+00:00,2026-04-07 23:13:43+00:00,0x7b6b6b07b04f5e49e127552056b795c3db7ef286


In [38]:
# check owner address, show all polymarket wallet it owns
OWNER_ADDRESS = "0x7b6b6b07b04f5e49e127552056b795c3db7ef286"

# sql ask user lookup for all polymarket wallets owned by this address
owner_wallet_sql = f"""
-- this query only checks ownership mapping, not trades
SELECT DISTINCT
    created_time,
    first_funded_time,
    has_been_funded,
    lower(concat('0x', to_hex(owner))) AS owner,
    lower(concat('0x', to_hex(polymarket_wallet))) AS wallet,
    wallet_type
FROM polymarket_polygon.users_address_lookup
WHERE lower(concat('0x', to_hex(owner))) = '{OWNER_ADDRESS}'
"""

# submit owner lookup to dune
response = requests.post(f"{DUNE_API_BASE}/sql/execute", headers=DUNE_HEADERS, json={"sql": owner_wallet_sql, "performance": DUNE_PERFORMANCE}, timeout=30)
execution_id = response.json()["execution_id"]

# small poll loop because dune runs async even for simple queries
state = ""
for _ in range(30):
    time.sleep(DUNE_POLL_INTERVAL_SECONDS)
    state = requests.get(f"{DUNE_API_BASE}/execution/{execution_id}/status", headers=DUNE_HEADERS, timeout=30).json()["state"]
    if state in {"QUERY_STATE_COMPLETED", "QUERY_STATE_COMPLETED_PARTIAL"}:
        break

# fetch the owned polymarket wallet rows
owner_wallets = pd.DataFrame(requests.get(
    f"{DUNE_API_BASE}/execution/{execution_id}/results",
    headers=DUNE_HEADERS,
    params={"allow_partial_results": "true", "limit": 10000},
    timeout=60,
).json()["result"]["rows"])

# clean date columns for display
owner_wallets["created_time"] = pd.to_datetime(owner_wallets["created_time"], errors="coerce", utc=True)
owner_wallets["first_funded_time"] = pd.to_datetime(owner_wallets["first_funded_time"], errors="coerce", utc=True)
owner_wallets = owner_wallets[["created_time", "first_funded_time", "has_been_funded", "owner", "wallet", "wallet_type"]]

print("Owner:", OWNER_ADDRESS)
print("Owned Polymarket wallets:", len(owner_wallets))

Owner: 0x7b6b6b07b04f5e49e127552056b795c3db7ef286
Owned Polymarket wallets: 1


## D4. Summary of key findings

This project shows that the Polymarket API can be used to study general market activity. At the platform level, trading is clearly concentrated in a small number of high-attention events and categories (see Q1-3). That said, the basic Polymarket API only gives open-event information. For deeper research, or anything closer to algorithmic trading, we need a better data source.

For my case study, I used the Dune API, a paid service. One thing that is different about Dune is that it runs queries asynchronously. After sending a request, Dune does not immediately return the final result, we need to check whether the query has finished, and then fetch the result once it is ready. The benefit is that Dune gives access to blockchain-level trading data, which made the wallet analysis possible.

Eventually, the data can flag suspicious insider-like behavior (see Q4-5), but it cannot prove insider trading by itself. On-chain data can show wallet timing, trade size, and directional bias, but it cannot reveal the real identity or intent behind a wallet. So I think the main value of this work is building a framework for spotting unusual trading activity that may deserve further investigation.

## D5. Further research

This analysis could be extended in a few useful directions.

- The abnormal-trading framework could be tested across more events instead of only one case. That would help build a baseline for what “normal” and “abnormal” trading behavior look like on Polymarket.
- The wallet-level analysis could be improved with more features, such as realized profit and loss, or market-price changes around key timestamps. This would help us better understand how prediction-market prices behave. It is still a new market, so there is a lot we do not fully understand yet.
- The project could become a more systematic monitoring tool. It could automatically detect unusual trading patterns, track the activity of flagged wallets, and even simulate those trades to make real profit!!!